In [ ]:
import numpy as np

# Rotation Matrices and Homogeneous Transforms


In [ ]:
Tx = lambda L,a: np.array([ [1,    0,          0,    L],
                          [0, np.cos(a), -np.sin(a), 0],
                          [0, np.sin(a),  np.cos(a), 0],
                          [0,    0,          0,      1]
                        ])
Ty = lambda L,a: np.array([[np.cos(a), 0, np.sin(a), 0],
                         [   0,      1,     0,      L],
                         [-np.sin(a), 0,  np.cos(a), 0],
                         [   0,      0,     0,      1]
                         ])
Tz = lambda L,a: np.array([[np.cos(a),-np.sin(a),0,  0],
                         [np.sin(a), np.cos(a),0,   0],
                         [   0,      0,     1,      L],
                         [   0,      0,     0,      1]
                         ])

Tyz = lambda L,a: Tx(0, -np.pi /2) @ Tz(L,a)

Txz = lambda L,a: Ty(0, np.pi /2) @ Tz(L,a)


Rx = lambda a: np.array([ [1,    0,          0],
                          [0, np.cos(a), -np.sin(a)],
                          [0, np.sin(a),  np.cos(a)],
                        ])
Ry = lambda a: np.array([[np.cos(a), 0, np.sin(a)],
                         [   0,      1,     0,     ],
                         [-np.sin(a), 0,  np.cos(a)],
                         ])
Rz = lambda a: np.array([[np.cos(a),-np.sin(a),0],
                         [np.sin(a), np.cos(a),0],
                         [   0,      0,     1,  ],
                         ])





def forward_kinematics(Lengths, axes, angles,):
  rot_dict = {'x':Tx,'y':Ty,'z':Tz,'P': lambda L, a: Tz(L + a, 0),}
  transforms = []
  T = np.eye(4)
  for L, a, ax in zip(Lengths, angles, axes):
    T = T@rot_dict[ax](L,a)
    transforms.append(T)

  return transforms












In [ ]:
x = [1.0, 0.5, 1.0]


print(Rz(x[0]) @ Ry(x[1]) @ Rx(x[2]))

[[ 0.47415988 -0.23667851  0.84803048]
 [ 0.73846026  0.63139506 -0.23667851]
 [-0.47942554  0.73846026  0.47415988]]


In [ ]:
def convert_T(q:np.ndarray):
  [x, y, z, roll, pitch, yaw] = q[:6]

  R =  Rz(yaw) @ Ry(pitch) @ Rx(roll)

  T = np.eye(4)
  T[:3,:3] = R
  T[:3,3] = [x,y,z]
  return T




In [ ]:
def compute_error(T_desired, T_current):

    p_err = T_desired[:3, 3] - T_current[:3, 3]


    R_err = T_desired[:3, :3] @ T_current[:3, :3].T

    angle = np.arccos(np.clip((np.trace(R_err) - 1) / 2, -1, 1))
    if np.isclose(angle, 0):
        w_err = np.zeros(3)
    else:
        w_err = angle / (2 * np.sin(angle)) * np.array([
            R_err[2,1] - R_err[1,2],
            R_err[0,2] - R_err[2,0],
            R_err[1,0] - R_err[0,1],
        ])

    return np.concatenate([p_err, w_err])

In [ ]:
def kinematics_jacobian(transforms, axes):
  axes_dict = {'x':0,'y':1,'z':2, 'P':2}
  n = len(transforms)
  J = np.zeros((6, n))

  p_e = transforms[-1][:3,3]
  for i,(T,ax) in enumerate(zip(transforms,axes)):
    hat = T[:3,axes_dict[ax]]
    if ax == 'P':
      J[:3, i] = hat
    else:
      p_i = T[:3,3]
      J[:3, i] = np.cross(hat, p_e - p_i)
      J[3:, i] = hat

  return J




In [ ]:
def kinematics(lengths, axes, angles, target, alpha=0.1,tol=1e-4,iters=200, pos_only=False):
  q = angles.copy()

  for _ in range(iters):
    transforms = forward_kinematics(lengths, axes, q)
    T_desired = convert_T(target)
    err = compute_error(T_desired, transforms[-1])
    if np.linalg.norm(err) < tol:
      break

    J = kinematics_jacobian(transforms, axes)
    if pos_only:
            J = J[:3]
            err = err[:3]
    dq = np.linalg.pinv(J) @ err
    q = q + alpha * dq

  transforms = forward_kinematics(lengths, axes, q)
  return  transforms, q


In [ ]:
def dif_kinematics(lengths, axes, angles, target, tol=1e-4, dt=1e-3, pos_only=False):
  q = angles.copy()


  transforms = forward_kinematics(lengths, axes, q)
  T_desired = convert_T(target)
  err = compute_error(T_desired,transforms[-1])
  if np.linalg.norm(err) < tol:
    return transforms, np.zeros_like(q)
  J = kinematics_jacobian(transforms, axes)
  if pos_only:
            J = J[:3]
            err = err[:3]
  dq = np.linalg.pinv(J) @ (err /dt)
  return transforms, dq


In [ ]:
L = [1,2,3,1,1,2,3]
axes = ['z','y','x','z','y','x','z']
a = [np.pi / 8, np.pi / 4,0.5,1.2, np.pi/2, np.pi /8, 0.8]
print(forward_kinematics(L, axes, a)[-1][:3,3])

transforms,q  = kinematics(L, axes, a,[0.85,3,-1,0,0,0], pos_only=True)
print(transforms[-1][:3,3])

[ 1.12669125  5.55326657 -1.6920457 ]
[ 0.85  3.   -1.  ]


#Dynamics


In [ ]:
def dynamics_jacobian(transforms, axes, coms):
  axes_dict = {'x': 0, 'y': 1, 'z': 2, 'P': 2}

  n = len(transforms)
  J_w = np.zeros((n,3,n))
  J_v = np.zeros((n,3,n))

  for i in range(n):

    p_ci = (transforms[i] @ np.append(coms[i], 1))[:3]

    for k in range(i+1):

      T_k = transforms[k]
      hat = T_k[:3, axes_dict[axes[k]]]

      if axes[k] == 'P':
        J_v[i,:,k] = hat
      else:
        p_k = T_k[:3,3]
        J_v[i,:,k] = np.cross(hat,p_ci - p_k)
        J_w[i,:,k] = hat

  return J_v, J_w



In [ ]:
def mass_matrix(masses, inertias, transforms, axes, coms):
  J_v, J_w = dynamics_jacobian(transforms, axes, coms)
  n = len(masses)
  M = np.zeros((n,n))

  for i in range(n):

    rot_i = transforms[i][:3,:3]
    m = masses[i]
    I_c = rot_i @ inertias[i] @ rot_i.T

    j_w = J_w[i]
    j_v  = J_v[i]

    M += m * j_v.T @ j_v + j_w.T @ I_c @ j_w

  return M



In [ ]:
# 1. Single-DOF ground truth
M = mass_matrix([1.0], [np.eye(3)], [np.eye(4)], ['z'], [np.zeros(3)])
print(M)  # [[1.0]]

# 2. Symmetry + positive definiteness on your 7-DOF setup
L = [1,2,3,1,1,2,3]
axes = ['z','y','x','z','y','x','z']
q = [np.pi/8, np.pi/4, 0.5, 1.2, np.pi/2, np.pi/8, 0.8]
masses = [1.0] * 7
inertias = [np.eye(3) * 0.1] * 7
coms = [np.zeros(3)] * 7

transforms = forward_kinematics(L, axes, q)
M = mass_matrix(masses, inertias, transforms, axes, coms)
print("symmetric:", np.allclose(M, M.T))
print("eigenvalues:", np.linalg.eigvalsh(M))  # all > 0

[[1.]]
symmetric: True
eigenvalues: [9.87650580e-02 4.15308304e-01 1.81305602e+00 1.06040445e+01
 1.35719265e+01 3.91936928e+01 9.97849685e+01]


In [ ]:
def derive_mass_matrix(masses, inertias, axes, coms, lengths, q, eps=1e-6):
  n = len(q)
  dM = np.zeros((n,n,n))
  for k in range(n):
    q_pos = q.copy(); q_pos[k] += eps
    q_neg = q.copy(); q_neg[k] -= eps
    T_pos = forward_kinematics(L, axes, q_pos)
    T_neg = forward_kinematics(L, axes, q_neg)
    M_pos = mass_matrix(masses, inertias, T_pos, axes, coms)
    M_neg = mass_matrix(masses, inertias, T_neg, axes, coms)
    dM[:,:,k] = (M_pos - M_neg) / (2 * eps)
  return dM

def coriolis_matrix(masses, inertias, axes, coms, lengths, q, dq):
    n = len(q)
    dM = derive_mass_matrix(masses, inertias, axes, coms, lengths, q)
    # Christoffel: Γ[i,j,k] = 0.5 * (dM[i,j,k] + dM[i,k,j] - dM[j,k,i])
    gamma = 0.5 * (dM + dM.transpose(0, 2, 1) - dM.transpose(1, 2, 0))
    # C[i,j] = sum_k gamma[i,j,k] * dq[k]
    C = gamma @ dq   # (n, n, n) @ (n,) → (n, n)
    return C

In [ ]:
def gravity_vector(masses, transforms, axes, coms, g_vec=np.array([0, 0, -9.81])):
    J_v, _ = dynamics_jacobian(transforms, axes, coms)
    n = len(masses)
    g = np.zeros(n)
    for j in range(n):

        F_grav = masses[j] * g_vec

        g -= J_v[j].T @ F_grav
    return g

In [ ]:
class RobotController:
    def __init__(self, freq, damping="critical"):
        omega = 2.0 * np.pi * freq

        self.kp = omega**2

        if damping == "critical":
            self.zeta = 1.0
        else:
            self.zeta = damping

        self.kv = 2.0 * self.zeta * omega

    def __call__(self, v, x, x_d, v_d, a):
        return self.kp * (x_d - x) + self.kv * (v_d -v) + a

# Complete Algorithm

In [ ]:
# robot config
L = [1, 1, 1, 0.5, 0.5, 0.3, 0.2]
axes = ['z', 'y', 'y', 'z', 'y', 'z', 'x']
q = np.array([0.0, 0.3, -0.5, 0.0, 0.4, 0.0, 0.0], dtype=float)
dq = np.zeros(7)
ddq = np.zeros(7)

masses = [2.0, 2.0, 1.5, 1.0, 0.5, 0.3, 0.2]
inertias = [np.eye(3) * 0.05 for _ in range(7)]
coms = [np.zeros(3) for _ in range(7)]

time_step = 1e-3
eps = 1e-4

# ---- Build a valid, reachable IK target ----
# 1. Pick a q slightly offset from current
q_goal = q + np.array([0.2, -0.15, 0.1, -0.1, 0.15, -0.1, 0.05])

# 2. Run FK on q_goal to get the corresponding EE pose
T_goal = forward_kinematics(L, axes, q_goal)
# Extract [x, y, z, roll, pitch, yaw] from T_goal — adjust to your FK's return format
# If forward_kinematics returns a list of transforms, take the last one:
T_ee = T_goal[-1] if isinstance(T_goal, list) else T_goal
pos = T_ee[:3, 3]
# Extract Euler angles (ZYX / RPY convention — adjust to match your kinematics)
from scipy.spatial.transform import Rotation as R
rpy = R.from_matrix(T_ee[:3, :3]).as_euler('xyz')
target = [*pos, *rpy]
print("target pose:", target)

# 3. Now IK should easily find q_desired ≈ q_goal
transforms, q_desired = kinematics(L, axes, q, target)
print("q_initial:", q)
print("q_goal   :", q_goal)
print("q_desired:", q_desired)
print("target reachable, initial error:", np.abs(q_desired - q).max())

dq_desired = np.zeros(7)
ddq_desired = np.zeros(7)

controller = RobotController(freq=5.0, damping='critical')

target pose: [np.float64(-0.2898201201478773), np.float64(2.4360235222984876), np.float64(1.7237013206509824), np.float64(0.04472902303077089), np.float64(0.3021338607488806), np.float64(0.006035253250951812)]
q_initial: [ 0.   0.3 -0.5  0.   0.4  0.   0. ]
q_goal   : [ 0.2   0.15 -0.4  -0.1   0.55 -0.1   0.05]
q_desired: [ 0.19995531  0.27497238 -0.52502762 -0.10002161  0.55003288 -0.09994316
  0.04996056]
target reachable, initial error: 0.19995531099776762


In [ ]:
#robot config
controller = RobotController(freq=1.0, damping='critical')

print(f'q_desired is: {q_desired}')
i = 0
while np.any(np.abs(q_desired - q) > eps) and i < 20000:

  transforms = forward_kinematics(L, axes, q)
  tau_prime = controller(dq, q, q_desired, dq_desired, ddq_desired)

  M = mass_matrix(masses, inertias, transforms, axes, coms)
  C = coriolis_matrix(masses, inertias,axes,coms,L, q, dq)
  G = gravity_vector(masses, transforms, axes, coms)

  tau = M @ tau_prime + C @ dq + G
  ddq = tau_prime
  dq += time_step * ddq
  q += dq * time_step
  i += 1
  if i % 500 == 0:
    print(f'Torque at timestep {i} : {tau}')
    print(f'joint acceleration at timestep {i} : {ddq}')
    print(f'joint velocity at timestep {i} : {dq}')
    print(f'joint position at timestep {i} : {q}')


q_desired is: [ 0.19995531  0.27497238 -0.52502762 -0.10002161  0.55003288 -0.09994316
  0.04996056]
Torque at timestep 500 : [-1.32784845e+01  2.94291850e+00  2.93835512e+00 -1.96062820e+00
 -6.28721308e-01  8.07201457e-03 -7.38725220e-03]
joint acceleration at timestep 500 : [-0.72917307  0.09126774  0.09126774  0.36474683 -0.54712192  0.36446073
 -0.18219019]
joint velocity at timestep 500 : [ 0.17003197 -0.02128224 -0.02128224 -0.08505337  0.12758044 -0.08498665
  0.04248396]
joint position at timestep 500 : [ 0.16424055  0.27944265 -0.52055735 -0.08215638  0.52323495 -0.08209194
  0.04103692]
Torque at timestep 1000 : [-1.42109588e+00  1.34498810e+00  1.34449946e+00 -1.25508017e+00
 -7.91004642e-01 -6.13385034e-04 -7.22779698e-04]
joint acceleration at timestep 1000 : [-0.07807827  0.00977275  0.00977275  0.0390563  -0.05858463  0.03902567
 -0.01950853]
joint velocity at timestep 1000 : [ 0.0148613  -0.00186013 -0.00186013 -0.00743392  0.01115091 -0.00742809
  0.00371323]
joint po